# <center>Evaluación de segmentaciones

In [ ]:
# Nombre de la carpeta que contiene las segmentaciones
segmentation_folder = 'segmentation'

# Nombre de carpeta que contiene la capa de referencia
evaluation_folder = 'evaluation'

#nombre de la capa de referencia
layer_ref = 'piscinas'

# importación de librerias 

In [25]:
import glob
import pandas as pd
import geopandas as gpd

# lectura de datos de segmentación

In [26]:
paths = glob.glob(f'{segmentation_folder}/*.shp')
gdfs = [gpd.read_file(path) for path in paths]

In [27]:
#información sobre los datos de segmentación
segmentation_names = []
for gdf, shp in zip(gdfs, paths):
    #número de objetos
    nObjetos = gdf.shape[0]

    #nombre de la segmentación
    name_file = shp.split('\\')[1].split('.')[0]
    segmentation_names.append(name_file)

    #cálculo del área de los objetos segmentados
    gdf['areaSeg'] = gdf.geometry.area

    #mostrando información
    print(f'\nSegmentación {name_file}: {nObjetos} objetos\n')
    print(f'\t{gdf.crs.to_string()}')
    print('\tcampos:\n', gdf.columns, '\n')



Segmentación beniseg40m20: 253725 objetos

	EPSG:23030
	campos:
 Index(['AREA', 'LENGTH', 'CONVEXITY', 'SOLIDITY', 'ROUNDNESS', 'FORMFACTOR',
       'ELONGATION', 'RECT_FIT', 'MAINDIR', 'MAJAXISLEN', 'MINAXISLEN',
       'HOLESOLRAT', 'areaSeg', 'geometry'],
      dtype='object') 


Segmentación beniseg60m10: 139346 objetos

	EPSG:23030
	campos:
 Index(['REGION_ID', 'AREA', 'LENGTH', 'COMPACT', 'CONVEXITY', 'SOLIDITY',
       'ROUNDNESS', 'FORMFACTOR', 'ELONGATION', 'RECT_FIT', 'MAINDIR',
       'MAJAXISLEN', 'MINAXISLEN', 'NUMHOLES', 'HOLESOLRAT', 'areaSeg',
       'geometry'],
      dtype='object') 


Segmentación beniseg75m20: 50142 objetos

	EPSG:23030
	campos:
 Index(['REGION_ID', 'areaSeg', 'geometry'], dtype='object') 



# lectura de datos de referencia

In [28]:
#lectura
pathRef = f'{evaluation_folder}/{layer_ref}.shp'
refObj = gpd.read_file(pathRef)

#cálculo del área de los objetos de referencia
refObj['areaObj'] = refObj.geometry.area

#cambiando el orden de los objetos en el geodataframe por nombre de objeto
refObj = refObj.sort_values(by='nombre')
refObj.head()

,id,nombre,areaObj,geometry
2,1,A,83.289307,"POLYGON ((755019.534 4275194.263, 755018.358 4..."
0,2,B,129.800203,"POLYGON ((755356.901 4275139.499, 755352.998 4..."
1,3,C,29.629819,"POLYGON ((755275.678 4274959.934, 755274.201 4..."


# Cálculo de índices

In [29]:
results = []
#iteración por cada segmentación
for segObj, segmentation_name in zip(gdfs, segmentation_names):
    #combrobación de sistemas de referencia espacial
    assert refObj.crs == segObj.crs, "capas en diferentes sistemas de referencia"

    #intersección entre objetos de referencia y la segmentación
    intersection = gpd.overlay(segObj, refObj, how='intersection')

    #creación de campos areaInter, areaX y areaY
    intersection['areaInter'] = intersection['geometry'].area
    intersection['areaX'] = intersection.areaInter / intersection.areaObj
    intersection['areaY'] = intersection.areaInter / intersection.areaSeg

    #guardando los poligonos de la intersección
    intersection.to_file(f'{evaluation_folder}/interseccion_{segmentation_name}.shp', driver='ESRI Shapefile')

    #iteración por cada objeto de referencia (A, B y C)
    for nombreObj in refObj.nombre:

        #selección de objetos segmentados para cada objeto de referencia
        mask = (intersection.nombre==nombreObj) & ((intersection.areaX > 0.5) | (intersection.areaY > 0.5))
        segObjSelected = intersection[mask]

        #guardando la selección
        segObjSelected.to_file(f'{evaluation_folder}/{segmentation_name}_{nombreObj}_intersection_select.shp', driver='ESRI Shapefile')

        #cálculo de áreas
        areaObj = segObjSelected.areaObj.unique()[0]
        sum_areaInter = segObjSelected.areaInter.sum()
        sum_areaSeg = segObjSelected.areaSeg.sum()
        max_areaSeg = segObjSelected.areaSeg.max()
        union_area = (sum_areaSeg + areaObj) - sum_areaInter

        #cálculo de índices
        error_defecto = (areaObj - sum_areaInter) / areaObj
        error_exceso = (sum_areaSeg - sum_areaInter) / areaObj
        AFI = (areaObj - max_areaSeg) / areaObj
        calidad = sum_areaInter / union_area

        #almacenando en memoria la información obtenida
        results.append({'segmentacion': segmentation_name, 'objeto': nombreObj, 'areaObj': areaObj,
                        '∑areaInter': sum_areaInter, '∑areaSeg': sum_areaSeg, 'max(AreaSeg)': max_areaSeg, 'unionArea': union_area,
                        'errorDefecto': error_defecto, 'errorExceso': error_exceso, 'AFI': AFI, 'indiceCalidad': calidad})

resultado = pd.DataFrame(results)

In [30]:
resultado.head(10)

,segmentacion,objeto,areaObj,∑areaInter,∑areaSeg,max(AreaSeg),unionArea,errorDefecto,errorExceso,AFI,indiceCalidad
0,beniseg40m20,A,83.289307,77.651815,84.0625,19.5000,89.699992,0.067686,0.076969,0.765876,0.865684
1,beniseg40m20,B,129.800203,124.391522,130.6250,57.0000,136.033681,0.041669,0.048024,0.560864,0.914417
2,beniseg40m20,C,29.629819,28.549991,32.0000,5.1250,33.079828,0.036444,0.116437,0.827032,0.863063
3,beniseg60m10,A,83.289307,76.379741,86.2500,70.8750,93.159566,0.082959,0.118506,0.149050,0.819881
4,beniseg60m10,B,129.800203,122.555319,128.2500,82.6250,135.494884,0.055816,0.043873,0.363445,0.904501
5,beniseg60m10,C,29.629819,28.497646,32.0000,16.0625,33.132173,0.038211,0.118204,0.457894,0.860120
6,beniseg75m20,A,83.289307,74.463249,656.9375,651.1875,665.763558,0.105969,6.993386,-6.818381,0.111846
7,beniseg75m20,B,129.800203,128.734108,374724.6250,374706.8750,374725.691095,0.008213,2885.942266,-2885.797304,0.000344
8,beniseg75m20,C,29.629819,28.709153,32.2500,27.8125,33.170667,0.031072,0.119503,0.061334,0.865498


# Cálculo de índices promedio

In [31]:
for segmentation_name in resultado.segmentacion.unique():
    print(f'=============={segmentation_name}==============\n============valores promedio============\n')
    print(resultado[resultado.segmentacion == segmentation_name].loc[:, 'errorDefecto':'indiceCalidad'].mean(),'\n')

==============beniseg40m20==============
============valores promedio============

errorDefecto     0.048600
errorExceso      0.080477
AFI              0.717924
indiceCalidad    0.881055
dtype: float64 

==============beniseg60m10==============
============valores promedio============

errorDefecto     0.058995
errorExceso      0.093527
AFI              0.323463
indiceCalidad    0.861501
dtype: float64 

==============beniseg75m20==============
============valores promedio============

errorDefecto       0.048418
errorExceso      964.351718
AFI             -964.184784
indiceCalidad      0.325896
dtype: float64 

